# EC2 Concrete Material Model

**Purpose**: Test and demonstrate the new EC2 concrete model implementation

**Status**: Development notebook

**Date**: January 10, 2026

## Overview

This notebook demonstrates the modern implementation of the EC2 concrete constitutive model:
- Uses new `BMCSModel` base class with Pydantic validation
- Symbolic expressions with SymPy
- Automatic UI generation for interactive exploration
- Proper parameter management and caching

## Setup

Import libraries and the new EC2 concrete model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint

# Import the new EC2 concrete model
from bmcs_cross_section.matmod.ec2_concrete import EC2Concrete

# Try to import UI adapters
try:
    from bmcs_cross_section.core.ui.jupyter import create_interactive_plot
    JUPYTER_UI_AVAILABLE = True
    print("✅ Jupyter UI available")
except ImportError as e:
    JUPYTER_UI_AVAILABLE = False
    print(f"⚠️ Jupyter UI not available: {e}")

print("✅ EC2 Concrete model imported successfully!")

## Test 1: Basic Model Creation

Create an EC2 concrete model with default parameters.

In [ ]:
# Create concrete model with f_cm = 28 MPa (default)
concrete = EC2Concrete()

print("EC2 Concrete Model")
print("=" * 50)
print("\nMaterial Properties:")
pprint(concrete.summary())

# Verify derived properties
print(f"\n✅ Model created successfully!")
print(f"   f_ck = {concrete.f_ck:.1f} MPa (characteristic strength)")
print(f"   E_cm = {concrete.E_cm:.0f} MPa (elastic modulus)")
print(f"   f_ctm = {concrete.f_ctm:.2f} MPa (tensile strength)")

## Test 2: Static Stress-Strain Plot

Plot the stress-strain curve without interactivity.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
concrete.plot_stress_strain(ax)
plt.tight_layout()
plt.show()

print("✅ Static plot generated!")

## Test 3: Different Concrete Grades

Test EC2 formulas for different concrete strength classes.

In [ ]:
# Test different concrete grades
concrete_grades = [
    ('C20/25', 20),
    ('C30/37', 30),
    ('C40/50', 40),
    ('C50/60', 50),
    ('C70/85', 70),
    ('C90/105', 90),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (grade_name, f_ck) in enumerate(concrete_grades):
    f_cm = f_ck + 8
    conc = EC2Concrete(f_cm=f_cm)
    
    ax = axes[idx]
    conc.plot_stress_strain(ax, n_points=300)
    ax.set_title(f'{grade_name}: $f_{{cm}}$ = {f_cm:.0f} MPa')

plt.tight_layout()
plt.show()

print("✅ All concrete grades plotted!")
print("\nKey observations:")
print("- Higher strength → higher peak stress")
print("- Higher strength → smaller ultimate strain (more brittle)")
print("- All follow EC2 parabolic-rectangular curve")

## Test 4: Compression Branch Detail

Examine the compression branch and EC2 formula validation.

In [ ]:
# Focus on compression branch
conc_c30 = EC2Concrete(f_cm=38.0)  # C30/37

# Generate compression strain range
eps_comp = np.linspace(1.5 * conc_c30.eps_cu1, 0, 300)
sig_comp = conc_c30.get_sig(eps_comp)
E_t_comp = conc_c30.get_E_t(eps_comp)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stress-strain
ax1.plot(eps_comp, sig_comp, 'b-', linewidth=2)
ax1.plot([conc_c30.eps_c1], [-conc_c30.f_cm], 'ro', markersize=10, 
         label=f'Peak: ε_c1={conc_c30.eps_c1:.4f}')
ax1.plot([conc_c30.eps_cu1], [conc_c30.get_sig(np.array([conc_c30.eps_cu1]))[0]], 
         'rs', markersize=10, label=f'Ultimate: ε_cu1={conc_c30.eps_cu1:.4f}')
ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax1.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax1.set_xlabel('Strain ε [-]')
ax1.set_ylabel('Stress σ [MPa]')
ax1.set_title('EC2 Compression Branch (C30/37)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Tangent modulus
ax2.plot(eps_comp, E_t_comp, 'g-', linewidth=2)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax2.axhline(y=conc_c30.E_cm, color='r', linestyle='--', linewidth=1, 
            label=f'E_cm = {conc_c30.E_cm:.0f} MPa')
ax2.set_xlabel('Strain ε [-]')
ax2.set_ylabel('Tangent Modulus E_t [MPa]')
ax2.set_title('Tangent Modulus Evolution')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Compression branch analysis complete!")
print(f"\nEC2 Parameters:")
print(f"  k = {conc_c30.k:.3f} (EC2 parameter)")
print(f"  E_cm = {conc_c30.E_cm:.0f} MPa")
print(f"  f_cm = {conc_c30.f_cm:.1f} MPa")

## Test 5: Tension Branch and Fiber Effects

Test the tension branch with different mu values (fiber reinforcement).

In [ ]:
# Compare different mu values
mu_values = [0.0, 0.3, 0.6, 1.0]

fig, ax = plt.subplots(figsize=(12, 7))

for mu in mu_values:
    conc = EC2Concrete(f_cm=38.0, mu=mu)
    
    # Generate tension strain range
    eps_tens = np.linspace(-0.0001, 0.001, 300)
    sig_tens = conc.get_sig(eps_tens)
    
    label = f'μ = {mu:.1f}'
    if mu == 0.0:
        label += ' (no fibers)'
    elif mu == 1.0:
        label += ' (full retention)'
    
    ax.plot(eps_tens, sig_tens, linewidth=2, label=label)

ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
ax.set_xlabel('Strain ε [-]')
ax.set_ylabel('Stress σ [MPa]')
ax.set_title('EC2 Tension Branch: Effect of Fiber Reinforcement (μ)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Fiber reinforcement effect demonstrated!")
print("\nμ parameter interpretation:")
print("  μ = 0.0: No post-crack strength (plain concrete)")
print("  μ = 1.0: Full tensile strength retained after cracking (ideal fibers)")
print("  0 < μ < 1: Partial strength retention (typical fiber-reinforced concrete)")

## Test 6: Parameter Validation

Test that Pydantic validation works correctly.

In [ ]:
from pydantic import ValidationError

print("Testing parameter validation...")

# Valid model
try:
    valid = EC2Concrete(f_cm=30.0, mu=0.5)
    print("✅ Valid parameters accepted")
except ValidationError as e:
    print(f"❌ Unexpected validation error: {e}")

# Test negative strength (should fail)
try:
    invalid = EC2Concrete(f_cm=-30.0)
    print("❌ Validation failed to catch negative f_cm!")
except ValidationError as e:
    print("✅ Validation caught negative f_cm:")
    print(f"   {e.errors()[0]['msg']}")

# Test out of range mu (should fail)
try:
    invalid = EC2Concrete(mu=1.5)
    print("❌ Validation failed to catch mu > 1!")
except ValidationError as e:
    print("✅ Validation caught mu > 1:")
    print(f"   {e.errors()[0]['msg']}")

# Test factor > 0
try:
    invalid = EC2Concrete(factor=0.0)
    print("❌ Validation failed to catch factor <= 0!")
except ValidationError as e:
    print("✅ Validation caught factor <= 0:")
    print(f"   {e.errors()[0]['msg']}")

print("\n✅ All validation tests passed!")

## Test 7: Cache Invalidation

Test that cached properties are properly invalidated when parameters change.

In [ ]:
# Create model and access cached property
conc = EC2Concrete(f_cm=30.0)
print(f"Initial f_ck: {conc.f_ck:.1f} MPa")
print(f"Initial E_cm: {conc.E_cm:.0f} MPa")

# Change parameter
conc.update_params(f_cm=50.0)
print(f"\nAfter update to f_cm=50:")
print(f"New f_ck: {conc.f_ck:.1f} MPa")
print(f"New E_cm: {conc.E_cm:.0f} MPa")

# Verify values changed correctly
assert conc.f_ck == 42.0, "f_ck not updated!"
assert abs(conc.E_cm - 35654) < 10, f"E_cm not updated! Expected ~35654, got {conc.E_cm:.1f}"

print("\n✅ Cache invalidation working correctly!")

## Test 8: Interactive Plot (if ipywidgets available)

Create an interactive plot with sliders for real-time parameter exploration.

In [ ]:
if JUPYTER_UI_AVAILABLE:
    # Create a fresh model for interactive exploration
    interactive_concrete = EC2Concrete(f_cm=38.0, mu=0.2)
    
    # Setup function (called once)
    def setup_plot(model, ax):
        """Setup axes and create artists"""
        ax.set_xlabel('Strain ε [-]')
        ax.set_ylabel('Stress σ [MPa]')
        ax.set_title('EC2 Concrete: Interactive Exploration')
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
        ax.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
        
        # Create artists
        line, = ax.plot([], [], 'b-', linewidth=2, label='σ-ε curve')
        peak_point, = ax.plot([], [], 'ro', markersize=10, label='Peak')
        crack_point, = ax.plot([], [], 'go', markersize=8, label='Crack')
        ax.legend()
        
        return {'line': line, 'peak': peak_point, 'crack': crack_point}
    
    # Update function (called on parameter change)
    def update_plot(model, ax, artists):
        """Update only the data"""
        # Generate data
        eps_min, eps_max = model.get_plot_range()
        eps = np.linspace(eps_min, eps_max, 500)
        sig = model.get_sig(eps)
        
        # Update line
        artists['line'].set_data(eps, sig)
        
        # Update key points
        artists['peak'].set_data([model.eps_c1], [-model.factor * model.f_cm])
        artists['crack'].set_data([model.eps_cr_computed], 
                                   [model.factor * model.f_ctm])
        
        # Update limits
        ax.set_xlim(eps_min, eps_max)
        ax.relim()
        ax.autoscale_view(scalex=False)
    
    # Create interactive plot
    create_interactive_plot(
        interactive_concrete,
        setup_plot,
        update_plot,
        figsize=(12, 7)
    )
    
    print("✅ Interactive plot created!")
    print("   Move the sliders to explore different parameters:")
    print("   - f_cm: compressive strength")
    print("   - mu: fiber reinforcement effect")
    print("   - factor: stress scaling")
else:
    # Fallback: static plot
    fig, ax = plt.subplots(figsize=(12, 7))
    interactive_concrete = EC2Concrete(f_cm=38.0)
    interactive_concrete.plot_stress_strain(ax)
    plt.tight_layout()
    plt.show()
    print("⚠️ Static plot shown (install ipywidgets for interactive version)")

## Test 9: Comparison with Legacy Implementation

Compare results with the old implementation (if available).

In [ ]:
# Try to import legacy model for comparison
try:
    from bmcs_cross_section.matmod.concrete.ec2_concrete_matmod import EC2ConcreteMatMod
    LEGACY_AVAILABLE = True
except ImportError:
    LEGACY_AVAILABLE = False
    print("⚠️ Legacy model not available for comparison")

if LEGACY_AVAILABLE:
    # Create both models with same parameters
    new_model = EC2Concrete(f_cm=38.0, mu=0.0)
    legacy_model = EC2ConcreteMatMod(f_cm=38.0, mu=0.0)
    
    # Generate test strains
    eps_test = np.linspace(-0.004, 0.0005, 500)
    
    # Compute stresses
    sig_new = new_model.get_sig(eps_test)
    sig_legacy = legacy_model.get_sig(eps_test)
    
    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Both curves
    ax1.plot(eps_test, sig_new, 'b-', linewidth=2, label='New Implementation')
    ax1.plot(eps_test, sig_legacy, 'r--', linewidth=2, label='Legacy Implementation')
    ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax1.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Strain ε [-]')
    ax1.set_ylabel('Stress σ [MPa]')
    ax1.set_title('EC2 Model Comparison: New vs Legacy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Difference
    diff = sig_new - sig_legacy
    ax2.plot(eps_test, diff, 'g-', linewidth=2)
    ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Strain ε [-]')
    ax2.set_ylabel('Difference [MPa]')
    ax2.set_title('Stress Difference (New - Legacy)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    max_diff = np.max(np.abs(diff))
    mean_diff = np.mean(np.abs(diff))
    
    print("✅ Comparison completed!")
    print(f"\nStatistics:")
    print(f"  Max absolute difference: {max_diff:.6f} MPa")
    print(f"  Mean absolute difference: {mean_diff:.6f} MPa")
    
    if max_diff < 0.01:
        print("  ✅ Excellent agreement (< 0.01 MPa)")
    elif max_diff < 0.1:
        print("  ✅ Good agreement (< 0.1 MPa)")
    else:
        print("  ⚠️ Notable differences detected")

## Summary

### What Works:
1. ✅ EC2 concrete model with Pydantic validation
2. ✅ Symbolic expression support with SymPy
3. ✅ Automatic parameter derivation from EC2 formulas
4. ✅ Cache invalidation on parameter updates
5. ✅ Interactive plotting with ipywidgets
6. ✅ Efficient plot updates (no flickering)
7. ✅ Proper error handling and validation
8. ✅ Comparison with legacy implementation

### Key Features:
- **Modern architecture**: Uses `BMCSModel` base class
- **Type safety**: Pydantic validation ensures valid parameters
- **Symbolic math**: SymPy expressions for derivatives
- **UI abstraction**: Works with Jupyter, Streamlit, etc.
- **Efficient**: Cached properties, lambdified functions
- **Clear API**: Simple `get_sig()`, `plot_stress_strain()` methods

### Next Steps:
1. Create similar models for:
   - Steel reinforcement (bilinear, multilinear)
   - Carbon reinforcement
   - Other concrete models (plateau, custom)
2. Integrate into mkappa application
3. Create validation test suite
4. Build Streamlit web application